In [1]:
import os
import glob
import rtree
import geopandas as gpd
import numpy as np
import pandas as pd
from osgeo import gdal
from shapely.geometry import Point
import matplotlib.pyplot as plt
from geopandas.tools import sjoin
from shapely.ops import nearest_points
from collections import defaultdict

### Step 1: Combine individual fire polygons with peak emissions, total emissions, etc 

In [15]:
directory='/Users/clairebekker/Library/CloudStorage/Box-Box/WBSE_Emissions/'

In [16]:
polys = gpd.read_file(directory+"daily_grid_CA/CA4195612355120180715_daily_emission_grid.shp")

In [33]:
#Sort all the shp files, and assign a number and get fire ID  
shp_files = sorted(glob.glob(directory+"daily_grid_CA/*.shp"),  key=os.path.getmtime)
shapes = []
fire = []
number = []
i = 1
for file in shp_files:
    #files = glob.glob('output/*')
    # Read Polygon Shapefile
    polys = gpd.read_file(file)
    polys = polys[polys['dob'] != -9999.0]
    polys = polys.fillna(0)
    # Reprojecting to 4326 coordinate system
    polys = polys.to_crs('EPSG:4326')
    name = file[78:99]
    polys['fire_id']= name
    fire.append(name)
    number.append(i)
    shapes.append(polys)
    i = i + 1
polys = pd.concat(shapes)
polys = gpd.GeoDataFrame(polys)

In [35]:
polys_dissolved = polys.dissolve(by='fire_id')

In [36]:
polys_dissolved = polys_dissolved [['geometry']]

In [37]:
data = pd.read_csv(directory+'Emissions_processing/fire_info_table.csv', sep=',')
data = data.rename(columns={"fire": "fire_id", "Peak Emissions":"Peak_Emissions", "Total Emissions":"Total_Emissions"})
#Combine data frames so that each fire polygon has peak and total emissions as well 
join = polys_dissolved.merge(data, on='fire_id')
#join = join.drop(columns=['Unnamed: 0']) #The index column 

In [38]:
join.head()

,fire_id,geometry,fire_name,longitude,latitude,First Day,Last Day,Peak Emissions Day,Peak_Emissions,Total_Emissions,Total Area (m^2),Low Severity Area (m^2),Medium Severity Area (m^2),High Severity Area (m^2),Low Severity (%),Medium Severity (%),High Severity (%),Duration
0,CA3338011743020180706,"MULTIPOLYGON (((-117.45911 33.36407, -117.4633...",NaN,-117.408870,33.365437,7/6/18,9/14/18,7/7/18,93700.0,209000.0,1199700,754020,401670,39240,62.850713,33.480870,3.270818,71
1,CA3367611751620180806,"POLYGON ((-117.52424 33.69773, -117.52543 33.7...",HOLY,-117.469863,33.697684,8/6/18,8/27/18,8/8/18,2180000.0,5350000.0,9246690,1489770,5352390,2403720,16.111387,57.884389,25.995464,22
2,CA3372011680020180725,"POLYGON ((-116.76022 33.69958, -116.76451 33.6...",CRANSTON,-116.696094,33.714858,7/25/18,7/28/18,7/26/18,1310000.0,2550000.0,5292360,1164510,3113190,985950,22.003605,58.824230,18.629685,4
3,CA3409611695820180706,"POLYGON ((-116.96558 34.08813, -116.96675 34.0...",NaN,-116.945753,34.105882,7/6/18,7/14/18,7/7/18,282000.0,356000.0,506700,116280,271620,114390,22.948490,53.605684,22.575488,9
4,CA3424011870020181108,"MULTIPOLYGON (((-118.94540 34.05423, -118.9466...",HILL,-118.797735,34.121373,11/9/18,11/25/18,11/9/18,9160000.0,11700000.0,39618270,9557730,21301020,8598330,24.124552,53.765649,21.702942,17


### Step 2: Determine in which air basin each fire resides in

In [40]:
air_basins = gpd.read_file('/Volumes/External drive/Summer_2022/site_selection/CA_air_basins/arb_california_airbasins_aligned_03.shp') #Current Projected CRS: EPSG:3310
air_basins = air_basins.set_geometry("geometry")
air_basins = air_basins.to_crs('EPSG:4326')
air_basins = air_basins.drop(columns=['AREA', 'PERIMETER'])
air_basins = air_basins.dissolve("BASIN_NAME").reset_index()
#join.to_file('MyGeometries.shp', driver='ESRI Shapefile')

In [42]:
largest_intersection = defaultdict(lambda: {'intersection_area': 0, 'BASIN_NAME': None, 
                                            'fire_id': None, 'Duration':0, 'Total_Emissions':0,
                                           'high_severity_%': 0})


# Iterate over each park and calculate the largest intersection area and associated air basin
for fire_index, fire_row in join.iterrows():
     #If Fire and Airbasins are both Multipolygon 
    if fire_row['geometry'].geom_type == 'MultiPolygon':
        for air_basin_index, air_basin_row in air_basins.iterrows():
            intersection_area = 0
            if air_basin_row['geometry'].geom_type == 'MultiPolygon':
                for polygon in fire_row['geometry'].geoms: 
                    for poly in air_basin_row['geometry'].geoms: 
                        area = polygon.intersection(poly).area
                        intersection_area = area + intersection_area
            #If Fire is a multipolygon and the airbasin is not 
            else: 
                for polygon in fire_row['geometry'].geoms:  # same for multipolygon.geoms
                    area = polygon.intersection(air_basin_row['geometry']).area
                    intersection_area = area + intersection_area
                    
            if intersection_area > largest_intersection[fire_index]['intersection_area']:
                largest_intersection[fire_index] = {
                'intersection_area': intersection_area,
                'BASIN_NAME': air_basin_row['BASIN_NAME'],
                'fire_id': fire_row["fire_id"], 
                'Duration': fire_row['Duration'], 
                'Total_Emissions': fire_row['Total_Emissions'],
                'high_severity_%': fire_row['High Severity (%)'],
            }     
                
    if fire_row['geometry'].geom_type == 'Polygon': 
        for air_basin_index, air_basin_row in air_basins.iterrows():
            intersection_area = 0
            #If Fire is not a multipolygon and the air basin is 
            if air_basin_row['geometry'].geom_type == 'MultiPolygon':
                for poly in air_basin_row['geometry'].geoms:
                    area = fire_row["geometry"].intersection(poly).area
                    intersection_area = area + intersection_area
            #If the Fire and Air Basin are not multipolygons 
            else: 
                intersection_area = fire_row["geometry"].intersection(air_basin_row['geometry']).area
            if intersection_area > largest_intersection[fire_index]['intersection_area']:
                largest_intersection[fire_index] = {
                'intersection_area': intersection_area,
                'BASIN_NAME': air_basin_row['BASIN_NAME'],
                'fire_id': fire_row["fire_id"], 
                'Duration': fire_row['Duration'], 
                'Total_Emissions': fire_row['Total_Emissions'],
                'high_severity_%': fire_row['High Severity (%)'],
            }     

In [43]:
largest_intersection_df = pd.DataFrame.from_dict(largest_intersection, orient='index')

In [44]:
largest_intersection_df.head(50)

,intersection_area,BASIN_NAME,fire_id,Duration,Total_Emissions,high_severity_%
0,0.002590,SAN DIEGO,CA3338011743020180706,71,209000.0,3.270818
1,0.012629,SOUTH COAST,CA3367611751620180806,22,5350000.0,25.995464
2,0.006827,SOUTH COAST,CA3372011680020180725,4,2550000.0,18.629685
3,0.001709,SOUTH COAST,CA3409611695820180706,9,356000.0,22.575488
4,0.028471,SOUTH COAST,CA3424011870020181108,17,11700000.0,21.702942
5,0.001886,SOUTH CENTRAL COAST,CA3424011893020181108,2,560000.0,2.140488
6,0.000153,SOUTH COAST,CA3449011860020180922,1,488000.0,9.467495
7,0.000508,SOUTH COAST,CA3490011851020180604,1,192000.0,0.495912
8,0.000643,SOUTH CENTRAL COAST,CA3511712010020180819,2,179000.0,6.931280
9,0.001427,SAN JOAQUIN VALLEY,CA3540011877020180803,2,71100.0,10.947698


### Step 3: Calculate Duration, High Severity, and Total Emissions for each Air Basin

In [45]:
duration = largest_intersection_df.groupby('BASIN_NAME', as_index=False)['Duration'].agg(['min', 'median', 'max']).reset_index()

In [46]:
high_sev = largest_intersection_df.groupby('BASIN_NAME', as_index=False)['high_severity_%'].agg(['mean']).reset_index()

In [47]:
total_emissions = largest_intersection_df.groupby('BASIN_NAME', as_index=False)['Total_Emissions'].agg(['sum']).reset_index()

In [48]:
#Merge duration, high_sev, total_emissions dataframes 
air_basins = air_basins.dissolve("BASIN_NAME").reset_index()
data = duration.merge(high_sev, on='BASIN_NAME').merge(total_emissions, on='BASIN_NAME')

In [49]:
#data = pd.concat([air_basins, data], join="outer", axis=1).reset_index()
data = pd.merge(air_basins, data,  how='left', left_on='BASIN_NAME', right_on='BASIN_NAME')
data = data.fillna(0)

In [50]:
data = data.rename(columns={"fire": "fire_id", "min":"Minimum Duration", "median":"Median Duration", "max": "Maximum Duration", "mean":"Average High Severity %", "sum":"Total Emissions"})
data = data.drop(columns=['ABASA_', 'ABASA_ID'])

In [51]:
data.head(15)

,BASIN_NAME,geometry,AB,index_x,Minimum Duration,Median Duration,Maximum Duration,index_y,Average High Severity %,index,Total Emissions
0,GREAT BASIN VALLEYS,"POLYGON ((-119.90419 38.93325, -119.89960 38.9...",GBV,0.0,1.0,2.5,4.0,0.0,12.795581,0.0,1828600.0
1,LAKE COUNTY,"POLYGON ((-122.88599 39.58020, -122.87799 39.5...",LC,1.0,12.0,30.0,48.0,1.0,28.092231,1.0,117300000.0
2,LAKE TAHOE,"POLYGON ((-120.20909 39.16161, -120.20906 39.1...",LT,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
3,MOJAVE DESERT,"POLYGON ((-118.79840 34.83201, -118.78500 34.8...",MD,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0
4,MOUNTAIN COUNTIES,"POLYGON ((-121.49350 40.43056, -121.49401 40.4...",MC,2.0,4.0,22.0,49.0,2.0,38.968283,2.0,42630400.0
5,NORTH CENTRAL COAST,"POLYGON ((-122.15203 37.28350, -122.15202 37.2...",NCC,3.0,1.0,1.5,2.0,3.0,8.860435,3.0,272300.0
6,NORTH COAST,"POLYGON ((-123.58863 41.90478, -123.58924 41.9...",NC,4.0,6.0,25.5,45.0,4.0,23.555060,4.0,1821000.0
7,NORTHEAST PLATEAU,"POLYGON ((-123.58863 41.90478, -123.58802 41.9...",NEP,5.0,2.0,6.0,74.0,5.0,18.362559,5.0,34098000.0
8,SACRAMENTO VALLEY,"POLYGON ((-121.33160 41.18455, -121.33160 41.1...",SV,6.0,2.0,16.0,68.0,6.0,23.789358,6.0,153871300.0
9,SALTON SEA,"POLYGON ((-115.33082 33.52149, -115.33006 33.5...",SS,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0


### Step 4:  Export Work

In [52]:
geo_data = gpd.GeoDataFrame(data, crs="EPSG:4326")
geo_data = geo_data.set_geometry("geometry")

In [21]:
geo_data.to_file('airbasins_and_stats.shp', driver='ESRI Shapefile')

/Users/marthasalazar/anaconda3/envs/fire/lib/python3.6/site-packages/ipykernel_launcher.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  """Entry point for launching an IPython kernel.


In [53]:
#To Display the table in map layout we want the excel file without geometry column
geo_data = data.drop(columns=['geometry'])
geo_data.to_csv(directory+"airbasins_and_stats.csv")  